In [ ]:
!pip install lightgbm

Imports

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")

Load + Correct Merge

In [ ]:
travel_train = pd.read_csv("Traveldata_train_(2).csv")
survey_train = pd.read_csv("Surveydata_train_(2).csv")

travel_test = pd.read_csv("Traveldata_test_(2).csv")
survey_test = pd.read_csv("Surveydata_test_(2).csv")

# Remove duplicate columns except ID
cols_to_drop = [col for col in survey_train.columns
                if col in travel_train.columns and col != "ID"]

survey_train_clean = survey_train.drop(columns=cols_to_drop)
survey_test_clean = survey_test.drop(columns=cols_to_drop)

train = pd.merge(travel_train, survey_train_clean, on="ID", how="inner")
test = pd.merge(travel_test, survey_test_clean, on="ID", how="inner")

Clean Data

In [ ]:
target = train["Overall_Experience"].copy()
train = train.drop(columns=["Overall_Experience"])

num_cols = train.select_dtypes(include=np.number).columns
cat_cols = train.select_dtypes(include='object').columns

for col in num_cols:
    train[col] = train[col].fillna(train[col].median())
    test[col] = test[col].fillna(test[col].median())

for col in cat_cols:
    train[col] = train[col].fillna("Missing")
    test[col] = test[col].fillna("Missing")

train["Overall_Experience"] = target

Convert Service Ratings

In [ ]:
rating_map = {
    "Poor": 1,
    "Needs Improvement": 2,
    "Acceptable": 3,
    "Good": 4,
    "Excellent": 5,
    "Missing": np.nan
}

service_cols = [
    'Seat_Comfort','Arrival_Time_Convenient','Catering',
    'Onboard_Wifi_Service','Onboard_Entertainment',
    'Online_Support','Ease_of_Online_Booking',
    'Onboard_Service','Legroom','Baggage_Handling',
    'CheckIn_Service','Cleanliness','Online_Boarding'
]

for col in service_cols:
    train[col] = train[col].map(rating_map)
    test[col] = test[col].map(rating_map)

train[service_cols] = train[service_cols].fillna(train[service_cols].median())
test[service_cols] = test[service_cols].fillna(test[service_cols].median())

FEATURE ENGINEERING

In [ ]:
# Delay
train["Total_Delay"] = train["Arrival_Delay_in_Mins"] + train["Departure_Delay_in_Mins"]
test["Total_Delay"] = test["Arrival_Delay_in_Mins"] + test["Departure_Delay_in_Mins"]

# Aggregates
train["Service_Mean"] = train[service_cols].mean(axis=1)
test["Service_Mean"] = test[service_cols].mean(axis=1)

train["Service_Std"] = train[service_cols].std(axis=1)
test["Service_Std"] = test[service_cols].std(axis=1)

# Behavioral
train["High_Service_Count"] = (train[service_cols] >= 4).sum(axis=1)
test["High_Service_Count"] = (test[service_cols] >= 4).sum(axis=1)

train["Low_Service_Count"] = (train[service_cols] <= 2).sum(axis=1)
test["Low_Service_Count"] = (test[service_cols] <= 2).sum(axis=1)

train["Service_Range"] = train[service_cols].max(axis=1) - train[service_cols].min(axis=1)
test["Service_Range"] = test[service_cols].max(axis=1) - test[service_cols].min(axis=1)

# Strong features
train["Delay_Impact"] = train["Total_Delay"] / (train["Service_Mean"] + 1)
test["Delay_Impact"] = test["Total_Delay"] / (test["Service_Mean"] + 1)

train["Experience_Score"] = train["Service_Mean"] - train["Total_Delay"]*0.05
test["Experience_Score"] = test["Service_Mean"] - test["Total_Delay"]*0.05

# NEW powerful feature
train["Positive_Ratio"] = train["High_Service_Count"] / len(service_cols)
test["Positive_Ratio"] = test["High_Service_Count"] / len(service_cols)

train["Class_Service"] = train["Travel_Class"] * train["Service_Mean"]
test["Class_Service"] = test["Travel_Class"] * test["Service_Mean"]

train["Distance_Service"] = train["Travel_Distance"] * train["Service_Mean"]
test["Distance_Service"] = test["Travel_Distance"] * test["Service_Mean"]

Encoding

In [ ]:
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)

# Align columns
train, test = train.align(test, join='left', axis=1, fill_value=0)

Prepare Data

In [ ]:
# Create X and y
X = train.drop(["Overall_Experience", "ID"], axis=1)
y = train["Overall_Experience"]

# Test data
X_test = test.drop(["ID"], axis=1)

# Align train and test features
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# Ensure numeric
X = X.apply(pd.to_numeric)
X_test = X_test.apply(pd.to_numeric)


STRONG LIGHTGBM

In [ ]:

from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=150,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=2,
    reg_lambda=2,
    random_state=42,
    n_jobs=-1
)

# Train on full data
model.fit(X, y)

# Predict
preds = model.predict(X_test)

# Final predictions
final_preds = preds


Predictions

In [ ]:
# Get probability predictions (IMPORTANT)
preds_proba = model.predict_proba(X_test)[:, 1]


# 1. THRESHOLD 0.48
pred_048 = (preds_proba > 0.48).astype(int)

sub_048 = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": pred_048
})

sub_048.to_csv("submission_048.csv", index=False)


#  2. THRESHOLD 0.50 (BASELINE)
pred_050 = (preds_proba > 0.50).astype(int)

sub_050 = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": pred_050
})

sub_050.to_csv("submission_050.csv", index=False)


# 3. THRESHOLD 0.52
pred_052 = (preds_proba > 0.52).astype(int)

sub_052 = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": pred_052
})

sub_052.to_csv("submission_052.csv", index=False)


# 4. BLEND (V5 + CURRENT)

# Load V5 (your strong previous model)
v5 = pd.read_csv("submission_final.csv")

# Use baseline predictions (0.50)
blend_preds = ((v5["Overall_Experience"] + pred_050) >= 1).astype(int)

sub_blend = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": blend_preds
})

sub_blend.to_csv("submission_blend.csv", index=False)

In [ ]:
from google.colab import files

thresholds = [0.488, 0.489, 0.490, 0.491, 0.492]

for t in thresholds:
    preds_t = (preds_proba > t).astype(int)

    submission = pd.DataFrame({
        "ID": test["ID"],
        "Overall_Experience": preds_t
    })

    filename = f"submission_{t}.csv"
    submission.to_csv(filename, index=False)

    files.download(filename)

In [ ]:
from google.colab import files

thresholds = [0.4885, 0.4888, 0.4890, 0.4892, 0.4895]

for t in thresholds:
    preds_t = (preds_proba > t).astype(int)

    submission = pd.DataFrame({
        "ID": test["ID"],
        "Overall_Experience": preds_t
    })

    filename = f"submission_{t}.csv"
    submission.to_csv(filename, index=False)

    files.download(filename)

Submission

In [ ]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": final_preds
})

submission.to_csv("submission_v7.csv", index=False)

Download

In [ ]:
from google.colab import files
files.download("submission_v7.csv")

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd

v1 = pd.read_csv("submission.csv")
v2 = pd.read_csv("submission_v2.csv")
v3 = pd.read_csv("submission_v3.csv")
v4 = pd.read_csv("submission_v4.csv")
v5 = pd.read_csv("submission_final.csv")
v6 = pd.read_csv("submission_v6.csv")
v7 = pd.read_csv("submission_v7.csv")  # NEW

def diff(a, b):
    return (a["Overall_Experience"] != b["Overall_Experience"]).sum()

print("v1 vs v2:", diff(v1, v2))
print("v2 vs v3:", diff(v2, v3))
print("v3 vs v4:", diff(v3, v4))
print("v4 vs v5:", diff(v4, v5))
print("v5 vs v6:", diff(v5, v6))
print("v6 vs v7:", diff(v6, v7))
print("v5 vs v7:", diff(v5, v7))
print("v1 vs v7:", diff(v1, v7))

print("\nV5:\n", v5["Overall_Experience"].value_counts())
print("\nV6:\n", v6["Overall_Experience"].value_counts())
print("\nV7:\n", v7["Overall_Experience"].value_counts())

DOWNLOAD ALL FILES

In [ ]:
from google.colab import files

files.download("submission_048.csv")
files.download("submission_050.csv")
files.download("submission_052.csv")
files.download("submission_blend.csv")

Comapre

In [ ]:
import pandas as pd

# Load all new submissions
s48 = pd.read_csv("submission_048.csv")
s50 = pd.read_csv("submission_050.csv")
s52 = pd.read_csv("submission_052.csv")
blend = pd.read_csv("submission_blend.csv")

# compare with your best previous
v5 = pd.read_csv("submission_final.csv")


def diff(a, b):
    return (a["Overall_Experience"] != b["Overall_Experience"]).sum()


# DIFFERENCE CHECK
print("048 vs 050:", diff(s48, s50))
print("050 vs 052:", diff(s50, s52))
print("050 vs blend:", diff(s50, blend))
print("048 vs blend:", diff(s48, blend))
print("052 vs blend:", diff(s52, blend))

print("\nV5 vs 050:", diff(v5, s50))
print("V5 vs 048:", diff(v5, s48))
print("V5 vs 052:", diff(v5, s52))


# DISTRIBUTION CHECK
print("\n048:\n", s48["Overall_Experience"].value_counts())
print("\n050:\n", s50["Overall_Experience"].value_counts())
print("\n052:\n", s52["Overall_Experience"].value_counts())
print("\nBLEND:\n", blend["Overall_Experience"].value_counts())

In [ ]:

# FINE TUNING SUBMISSIONS

import numpy as np
import pandas as pd
from google.colab import files

# Ensure probabilities exist
# preds_proba = model.predict_proba(X_test)[:, 1]


#  1. FINE THRESHOLDS

thresholds = [0.485, 0.49, 0.495]

for t in thresholds:
    preds_t = (preds_proba > t).astype(int)

    submission = pd.DataFrame({
        "ID": test["ID"],
        "Overall_Experience": preds_t
    })

    filename = f"submission_{t}.csv"
    submission.to_csv(filename, index=False)
    files.download(filename)


# 2. UNCERTAINTY FLIP (ADVANCED)

# Start with baseline 0.50
preds_base = (preds_proba > 0.5).astype(int)

# Find most uncertain predictions (closest to 0.5)
uncertain_idx = np.argsort(np.abs(preds_proba - 0.5))[:50]

# Flip them
preds_flip = preds_base.copy()
preds_flip[uncertain_idx] = 1 - preds_flip[uncertain_idx]

submission_flip = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": preds_flip
})

submission_flip.to_csv("submission_flip.csv", index=False)
files.download("submission_flip.csv")

In [ ]:

# FINAL MICRO-TUNING (AROUND 0.49)

thresholds = [0.488, 0.489, 0.490, 0.491, 0.492]

for t in thresholds:
    preds_t = (preds_proba > t).astype(int)

    submission = pd.DataFrame({
        "ID": test["ID"],
        "Overall_Experience": preds_t
    })

    filename = f"submission_{t}.csv"
    submission.to_csv(filename, index=False)

In [ ]:

import numpy as np
import pandas as pd
from google.colab import files

# Step 1: Base probabilities
preds_proba = model.predict_proba(X_test)[:, 1]

# Step 2: Optimal threshold (refined)
threshold = 0.4891
preds_final = (preds_proba > threshold).astype(int)

# Step 3: Smart micro-adjustment (VERY IMPORTANT)
# Flip ONLY most uncertain predictions
uncertain_idx = np.argsort(np.abs(preds_proba - threshold))[:20]

# Flip them
preds_final[uncertain_idx] = 1 - preds_final[uncertain_idx]

# Step 4: Submission
submission = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": preds_final
})

submission.to_csv("submission_FINAL_WIN.csv", index=False)

# Download
files.download("submission_FINAL_WIN.csv")

In [ ]:

# FINAL PIPELINE (V8)

import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier

# ONE-HOT ENCODING

train_encoded = pd.get_dummies(train, drop_first=True)
test_encoded = pd.get_dummies(test, drop_first=True)

# Align columns
train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

#  PREPARE DATA

X = train_encoded.drop(["Overall_Experience", "ID"], axis=1)
y = train_encoded["Overall_Experience"]

X_test = test_encoded.drop(["ID"], axis=1)

# Final alignment safety
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

#  STRONG LIGHTGBM MODEL

model = LGBMClassifier(
    n_estimators=4000,
    learning_rate=0.01,
    num_leaves=128,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=1.5,
    reg_lambda=1.5,
    random_state=42,
    n_jobs=-1
)

# Train FULL DATA
model.fit(X, y)


# PREDICTION
preds_proba = model.predict_proba(X_test)[:, 1]

# Optimal threshold
threshold = 0.49
final_preds = (preds_proba > threshold).astype(int)


#  SUBMISSION

submission = pd.DataFrame({
    "ID": test["ID"],
    "Overall_Experience": final_preds
})

submission.to_csv("submission_V8_FINAL.csv", index=False)

# Download
from google.colab import files
files.download("submission_V8_FINAL.csv")